# Reward Model Training for RLHF

## Overview

Reward modeling is the first step in Reinforcement Learning from Human Feedback (RLHF), introduced by:
- **Christiano et al. (2017)**: "Deep reinforcement learning from human preferences"
- **Stiennon et al. (2020)**: "Learning to summarize with human feedback" (OpenAI)
- **Ouyang et al. (2022)**: "Training language models to follow instructions with human feedback" (InstructGPT)

### Key Insight
Instead of defining a reward function manually, we train a model to predict human preferences, then use it to guide policy optimization.

### RLHF Pipeline
1. **Supervised Fine-Tuning (SFT)**: Train base model on demonstrations
2. **Reward Modeling**: Train model to predict human preferences
3. **RL Fine-Tuning**: Optimize policy using reward model (PPO)

### Historical Context
- **2017**: Deep RL from human preferences (Atari games)
- **2020**: Applied to text summarization
- **2022**: InstructGPT scales RLHF to 175B parameters
- **2023**: ChatGPT popularizes RLHF alignment

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from tqdm import tqdm
import math

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Reward Model Architecture

### Design Principles

A reward model takes (prompt, completion) pairs and outputs a scalar reward score.

**Architecture**: Language model + value head
```
Input: [prompt] [completion] [EOS]
         ↓
    Transformer (frozen or fine-tuned)
         ↓
    Last token embedding
         ↓
    Linear projection
         ↓
    Scalar reward
```

### Key Design Decisions
1. **Base model**: Usually same architecture as policy model
2. **Value head**: Single linear layer (simple) or MLP (more capacity)
3. **Pooling**: Last token (standard) vs mean pooling
4. **Normalization**: Reward scaling for stability

In [ ]:
class RewardModel(nn.Module):
    """Reward model with transformer base + value head."""
    
    def __init__(
        self,
        vocab_size: int,
        hidden_dim: int = 512,
        num_layers: int = 6,
        num_heads: int = 8,
        max_seq_len: int = 512,
        dropout: float = 0.1,
        value_head_type: str = 'linear'  # 'linear' or 'mlp'
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Transformer backbone
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=4 * hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Value head
        if value_head_type == 'linear':
            self.value_head = nn.Linear(hidden_dim, 1)
        else:  # MLP
            self.value_head = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1)
            )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Args:
            input_ids: (batch_size, seq_len)
            attention_mask: (batch_size, seq_len)
        
        Returns:
            rewards: (batch_size,) scalar rewards
        """
        batch_size, seq_len = input_ids.shape
        
        # Embeddings
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        x = self.dropout(x)
        
        # Transformer
        if attention_mask is not None:
            # Convert attention mask to transformer format
            attention_mask = attention_mask.bool()
        
        x = self.transformer(x, src_key_padding_mask=~attention_mask if attention_mask is not None else None)
        
        # Extract last token representation
        if attention_mask is not None:
            # Get last non-padded token for each sequence
            last_token_indices = attention_mask.sum(dim=1) - 1
            last_hidden = x[torch.arange(batch_size), last_token_indices]
        else:
            last_hidden = x[:, -1, :]  # Last token
        
        # Value head
        rewards = self.value_head(last_hidden).squeeze(-1)
        
        return rewards


# Test reward model
model = RewardModel(vocab_size=1000, hidden_dim=256, num_layers=4)
print(f"Reward Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
batch_size, seq_len = 4, 64
input_ids = torch.randint(0, 1000, (batch_size, seq_len))
attention_mask = torch.ones(batch_size, seq_len)

rewards = model(input_ids, attention_mask)
print(f"\nInput shape: {input_ids.shape}")
print(f"Output rewards shape: {rewards.shape}")
print(f"Sample rewards: {rewards}")

## 2. Preference Data Collection

### Data Format

Preference data consists of comparisons:
```
(prompt, completion_chosen, completion_rejected)
```

### Collection Methods

1. **Human Labeling**
   - Show two completions, ask "which is better?"
   - Can add confidence levels
   - Expensive but high quality

2. **AI Labeling**
   - Use GPT-4 to judge preferences
   - Constitutional AI: model judges its own outputs
   - Cheaper but may have biases

3. **Implicit Feedback**
   - User engagement (clicks, time spent)
   - Revision patterns
   - Noisy but scalable

### Quality Criteria
- **Helpfulness**: Does it answer the question?
- **Harmlessness**: Is it safe and ethical?
- **Honesty**: Is it truthful and well-calibrated?
- **Coherence**: Is it well-written and logical?

In [ ]:
@dataclass
class PreferenceExample:
    """Single preference comparison."""
    prompt: str
    chosen: str
    rejected: str
    margin: float = 1.0  # Confidence/strength of preference


class PreferenceDataset(Dataset):
    """Dataset for preference comparisons."""
    
    def __init__(
        self,
        examples: List[PreferenceExample],
        tokenizer: Optional[callable] = None,
        max_length: int = 512
    ):
        self.examples = examples
        self.tokenizer = tokenizer or self._simple_tokenizer
        self.max_length = max_length
    
    def _simple_tokenizer(self, text: str) -> List[int]:
        """Simple character-level tokenizer for demo."""
        return [ord(c) % 1000 for c in text[:self.max_length]]
    
    def __len__(self):
        return len(self.examples)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        example = self.examples[idx]
        
        # Tokenize chosen and rejected completions
        chosen_text = example.prompt + " " + example.chosen
        rejected_text = example.prompt + " " + example.rejected
        
        chosen_ids = self.tokenizer(chosen_text)
        rejected_ids = self.tokenizer(rejected_text)
        
        # Pad to same length
        max_len = max(len(chosen_ids), len(rejected_ids))
        chosen_ids = chosen_ids + [0] * (max_len - len(chosen_ids))
        rejected_ids = rejected_ids + [0] * (max_len - len(rejected_ids))
        
        return {
            'chosen_input_ids': torch.tensor(chosen_ids),
            'rejected_input_ids': torch.tensor(rejected_ids),
            'chosen_attention_mask': torch.tensor([1] * len(chosen_ids)),
            'rejected_attention_mask': torch.tensor([1] * len(rejected_ids)),
            'margin': torch.tensor(example.margin)
        }


# Create synthetic preference dataset
def create_preference_dataset(num_examples: int = 1000) -> List[PreferenceExample]:
    """Create synthetic preference data for demonstration."""
    examples = []
    
    prompts = [
        "Explain photosynthesis:",
        "Write a poem about mountains:",
        "How do I bake cookies?",
        "What is quantum computing?",
        "Summarize World War 2:"
    ]
    
    for i in range(num_examples):
        prompt = np.random.choice(prompts)
        
        # Simulate chosen (better) and rejected (worse) completions
        chosen = f"High quality response {i} that is detailed and accurate."
        rejected = f"Low quality response {i} that is brief."
        
        margin = np.random.uniform(0.5, 2.0)  # Preference strength
        
        examples.append(PreferenceExample(prompt, chosen, rejected, margin))
    
    return examples


# Create dataset
preference_examples = create_preference_dataset(1000)
dataset = PreferenceDataset(preference_examples)

print(f"Created {len(dataset)} preference examples")
print(f"\nSample example:")
sample = preference_examples[0]
print(f"Prompt: {sample.prompt}")
print(f"Chosen: {sample.chosen}")
print(f"Rejected: {sample.rejected}")
print(f"Margin: {sample.margin:.2f}")

## 3. Bradley-Terry Model

### Probabilistic Framework

The Bradley-Terry model (1952) provides a principled way to learn from pairwise comparisons.

### Mathematical Formulation

Given two completions $y_1$ and $y_2$ for prompt $x$, the probability that $y_1$ is preferred:

$$P(y_1 \succ y_2 | x) = \frac{\exp(r_\theta(x, y_1))}{\exp(r_\theta(x, y_1)) + \exp(r_\theta(x, y_2))}$$

This simplifies to:

$$P(y_1 \succ y_2 | x) = \sigma(r_\theta(x, y_1) - r_\theta(x, y_2))$$

where $\sigma$ is the sigmoid function and $r_\theta(x, y)$ is the reward model.

### Loss Function

The negative log-likelihood loss:

$$\mathcal{L}(\theta) = -\mathbb{E}_{(x, y_w, y_l) \sim D} \left[ \log \sigma(r_\theta(x, y_w) - r_\theta(x, y_l)) \right]$$

where:
- $y_w$ is the preferred (winning) completion
- $y_l$ is the rejected (losing) completion
- $D$ is the preference dataset

### Properties
1. **Invariant to reward scaling**: Only differences matter
2. **Well-calibrated**: Outputs valid probabilities
3. **Simple**: Single sigmoid comparison

In [ ]:
class BradleyTerryLoss(nn.Module):
    """Bradley-Terry loss for preference learning."""
    
    def __init__(self, margin_scale: float = 1.0):
        super().__init__()
        self.margin_scale = margin_scale
    
    def forward(
        self,
        reward_chosen: torch.Tensor,
        reward_rejected: torch.Tensor,
        margin: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Args:
            reward_chosen: (batch_size,) rewards for chosen completions
            reward_rejected: (batch_size,) rewards for rejected completions
            margin: (batch_size,) optional preference strength
        
        Returns:
            loss: scalar loss
        """
        # Bradley-Terry: -log sigmoid(r_chosen - r_rejected)
        logits = reward_chosen - reward_rejected
        
        if margin is not None:
            # Scale by margin (stronger preferences => larger gradient)
            logits = logits * margin * self.margin_scale
        
        # Negative log probability
        loss = -F.logsigmoid(logits).mean()
        
        return loss


# Visualize Bradley-Terry probability
def visualize_bradley_terry():
    reward_diffs = np.linspace(-5, 5, 100)
    probs = 1 / (1 + np.exp(-reward_diffs))
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Probability curve
    ax1.plot(reward_diffs, probs, linewidth=2)
    ax1.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Equal preference')
    ax1.axvline(x=0, color='r', linestyle='--', alpha=0.5)
    ax1.set_xlabel('Reward Difference (r_chosen - r_rejected)', fontsize=12)
    ax1.set_ylabel('P(chosen > rejected)', fontsize=12)
    ax1.set_title('Bradley-Terry Probability Function', fontsize=14)
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Loss landscape
    losses = -np.log(probs + 1e-10)
    ax2.plot(reward_diffs, losses, linewidth=2, color='orange')
    ax2.set_xlabel('Reward Difference (r_chosen - r_rejected)', fontsize=12)
    ax2.set_ylabel('Loss (-log probability)', fontsize=12)
    ax2.set_title('Bradley-Terry Loss Landscape', fontsize=14)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 5)
    
    plt.tight_layout()
    plt.show()
    
    print("Key observations:")
    print("1. Sigmoid shape: smooth probability transition")
    print("2. Loss decreases as reward difference increases")
    print("3. Symmetric around r_chosen = r_rejected")
    print("4. Gradient vanishes when difference is large (confident predictions)")

visualize_bradley_terry()

## 4. Training the Reward Model

### Training Procedure

1. **Initialize**: Start from pre-trained language model
2. **Add value head**: Small linear layer or MLP
3. **Train on comparisons**: Minimize Bradley-Terry loss
4. **Regularization**: Prevent overfitting to labeler preferences

### Hyperparameters
- **Learning rate**: Typically 1e-5 to 1e-4
- **Batch size**: 32-128 comparisons
- **Epochs**: 1-5 (careful not to overfit)
- **Weight decay**: 0.01-0.1

In [ ]:
def train_reward_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: Optional[DataLoader] = None,
    num_epochs: int = 3,
    learning_rate: float = 1e-4,
    weight_decay: float = 0.01,
    device: torch.device = torch.device('cpu')
) -> Dict[str, List[float]]:
    """Train reward model on preference data."""
    
    model = model.to(device)
    criterion = BradleyTerryLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    
    history = {'train_loss': [], 'val_loss': [], 'accuracy': []}
    
    for epoch in range(num_epochs):
        # Training
        model.train()
        train_loss = 0.0
        correct = 0
        total = 0
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}')
        for batch in pbar:
            # Move to device
            chosen_ids = batch['chosen_input_ids'].to(device)
            rejected_ids = batch['rejected_input_ids'].to(device)
            chosen_mask = batch['chosen_attention_mask'].to(device)
            rejected_mask = batch['rejected_attention_mask'].to(device)
            margin = batch['margin'].to(device)
            
            # Forward pass
            reward_chosen = model(chosen_ids, chosen_mask)
            reward_rejected = model(rejected_ids, rejected_mask)
            
            # Compute loss
            loss = criterion(reward_chosen, reward_rejected, margin)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            # Metrics
            train_loss += loss.item()
            correct += (reward_chosen > reward_rejected).sum().item()
            total += len(reward_chosen)
            
            pbar.set_postfix({
                'loss': loss.item(),
                'acc': f'{100 * correct / total:.1f}%'
            })
        
        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = correct / total
        history['train_loss'].append(avg_train_loss)
        history['accuracy'].append(train_accuracy)
        
        # Validation
        if val_loader is not None:
            val_loss, val_acc = evaluate_reward_model(model, val_loader, criterion, device)
            history['val_loss'].append(val_loss)
            print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, "
                  f"Val Loss={val_loss:.4f}, Train Acc={train_accuracy:.4f}, Val Acc={val_acc:.4f}")
        else:
            print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, Train Acc={train_accuracy:.4f}")
    
    return history


def evaluate_reward_model(
    model: nn.Module,
    data_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device
) -> Tuple[float, float]:
    """Evaluate reward model."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in data_loader:
            chosen_ids = batch['chosen_input_ids'].to(device)
            rejected_ids = batch['rejected_input_ids'].to(device)
            chosen_mask = batch['chosen_attention_mask'].to(device)
            rejected_mask = batch['rejected_attention_mask'].to(device)
            margin = batch['margin'].to(device)
            
            reward_chosen = model(chosen_ids, chosen_mask)
            reward_rejected = model(rejected_ids, rejected_mask)
            
            loss = criterion(reward_chosen, reward_rejected, margin)
            total_loss += loss.item()
            
            correct += (reward_chosen > reward_rejected).sum().item()
            total += len(reward_chosen)
    
    return total_loss / len(data_loader), correct / total


# Train model
print("Training reward model on synthetic data...\n")

# Split data
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

# Initialize model
reward_model = RewardModel(vocab_size=1000, hidden_dim=128, num_layers=3)

# Train
history = train_reward_model(
    reward_model,
    train_loader,
    val_loader,
    num_epochs=3,
    learning_rate=1e-4,
    device=device
)

In [ ]:
# Plot training curves
def plot_training_history(history: Dict[str, List[float]]):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    epochs = range(1, len(history['train_loss']) + 1)
    ax1.plot(epochs, history['train_loss'], 'o-', label='Train Loss', linewidth=2)
    if 'val_loss' in history and history['val_loss']:
        ax1.plot(epochs, history['val_loss'], 's-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curve
    ax2.plot(epochs, history['accuracy'], 'o-', color='green', linewidth=2)
    ax2.axhline(y=0.5, color='r', linestyle='--', alpha=0.5, label='Random baseline')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title('Training Accuracy', fontsize=14)
    ax2.set_ylim(0, 1)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_training_history(history)

## 5. Validation Strategies

### Evaluation Metrics

1. **Preference Accuracy**
   - % of comparisons correctly ranked
   - Should be >> 50% (random baseline)

2. **Agreement with Humans**
   - Hold-out test set of human preferences
   - Measure correlation with human judgments

3. **Reward Distribution Analysis**
   - Check for reasonable reward ranges
   - Look for degenerate solutions (all same reward)

4. **Out-of-Distribution Robustness**
   - Test on different prompts/domains
   - Check for unexpected behaviors

### Cross-Validation
- Split by prompt (not by comparison)
- Ensures generalization to new prompts
- More realistic evaluation

In [ ]:
def analyze_reward_distribution(
    model: nn.Module,
    dataset: Dataset,
    device: torch.device,
    num_samples: int = 200
):
    """Analyze distribution of predicted rewards."""
    model.eval()
    
    chosen_rewards = []
    rejected_rewards = []
    reward_diffs = []
    
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            batch = dataset[i]
            
            chosen_ids = batch['chosen_input_ids'].unsqueeze(0).to(device)
            rejected_ids = batch['rejected_input_ids'].unsqueeze(0).to(device)
            chosen_mask = batch['chosen_attention_mask'].unsqueeze(0).to(device)
            rejected_mask = batch['rejected_attention_mask'].unsqueeze(0).to(device)
            
            r_chosen = model(chosen_ids, chosen_mask).item()
            r_rejected = model(rejected_ids, rejected_mask).item()
            
            chosen_rewards.append(r_chosen)
            rejected_rewards.append(r_rejected)
            reward_diffs.append(r_chosen - r_rejected)
    
    # Plot distributions
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Chosen vs Rejected rewards
    axes[0, 0].hist(chosen_rewards, bins=30, alpha=0.6, label='Chosen', color='green')
    axes[0, 0].hist(rejected_rewards, bins=30, alpha=0.6, label='Rejected', color='red')
    axes[0, 0].set_xlabel('Reward', fontsize=12)
    axes[0, 0].set_ylabel('Count', fontsize=12)
    axes[0, 0].set_title('Reward Distributions', fontsize=14)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Reward differences
    axes[0, 1].hist(reward_diffs, bins=30, color='blue', alpha=0.7)
    axes[0, 1].axvline(x=0, color='r', linestyle='--', linewidth=2, label='No preference')
    axes[0, 1].set_xlabel('Reward Difference (chosen - rejected)', fontsize=12)
    axes[0, 1].set_ylabel('Count', fontsize=12)
    axes[0, 1].set_title('Reward Difference Distribution', fontsize=14)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Scatter plot
    axes[1, 0].scatter(chosen_rewards, rejected_rewards, alpha=0.5)
    axes[1, 0].plot([min(chosen_rewards), max(chosen_rewards)], 
                     [min(chosen_rewards), max(chosen_rewards)],
                     'r--', linewidth=2, label='Equal rewards')
    axes[1, 0].set_xlabel('Chosen Reward', fontsize=12)
    axes[1, 0].set_ylabel('Rejected Reward', fontsize=12)
    axes[1, 0].set_title('Chosen vs Rejected Rewards', fontsize=14)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Statistics
    stats_text = f"""
    Chosen Rewards:
      Mean: {np.mean(chosen_rewards):.3f}
      Std: {np.std(chosen_rewards):.3f}
    
    Rejected Rewards:
      Mean: {np.mean(rejected_rewards):.3f}
      Std: {np.std(rejected_rewards):.3f}
    
    Reward Differences:
      Mean: {np.mean(reward_diffs):.3f}
      Std: {np.std(reward_diffs):.3f}
      % Positive: {100 * np.mean(np.array(reward_diffs) > 0):.1f}%
    """
    axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
                     verticalalignment='center')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print analysis
    print("\n=== Reward Model Analysis ===")
    print(f"Samples analyzed: {len(chosen_rewards)}")
    print(f"\nAccuracy (% correct rankings): {100 * np.mean(np.array(reward_diffs) > 0):.1f}%")
    print(f"Mean reward difference: {np.mean(reward_diffs):.3f}")
    
    if np.mean(reward_diffs) < 0.01:
        print("\nWARNING: Very small reward differences. Model may not be discriminative.")
    if np.std(chosen_rewards) < 0.01:
        print("\nWARNING: Reward variance very low. Model may have collapsed.")


# Analyze trained model
analyze_reward_distribution(reward_model, dataset, device)

## 6. Common Failure Modes

### 1. Reward Hacking
**Problem**: Policy exploits reward model flaws
- Produces nonsensical but high-reward outputs
- Exploits biases in preference data

**Solutions**:
- KL divergence penalty (keep policy close to reference)
- Diverse preference data
- Ensemble reward models

### 2. Overfitting to Labelers
**Problem**: Model learns labeler quirks, not true quality
- Doesn't generalize to other users
- Learns superficial patterns

**Solutions**:
- Multiple labelers per comparison
- Regularization
- Early stopping

### 3. Reward Collapse
**Problem**: All outputs get similar rewards
- No gradient signal for RL
- Model can't distinguish good from bad

**Solutions**:
- Larger model capacity
- Better initialization
- Careful learning rate tuning

### 4. Distribution Shift
**Problem**: RL policy generates OOD samples
- Reward model uncertain on new data
- Can give spurious high rewards

**Solutions**:
- Iterative data collection (feedback loop)
- Uncertainty quantification
- Conservative reward estimates

### 5. Alignment Taxes
**Problem**: Alignment reduces capability
- Trade-off between safety and usefulness
- May hurt performance on benchmarks

**Solutions**:
- Careful data curation
- Multi-objective optimization
- Task-specific fine-tuning after alignment

In [ ]:
# Simulate reward hacking scenario
def simulate_reward_hacking():
    """Demonstrate how policy can exploit reward model."""
    
    # Create simple reward model with known flaw
    # (rewards longer responses, regardless of quality)
    def flawed_reward(text: str) -> float:
        """Reward model that just counts length."""
        return len(text) / 100.0
    
    # Simulate policy optimization
    texts = [
        "Good answer.",  # Short, good quality
        "This is a detailed and comprehensive answer.",  # Medium
        "x" * 1000  # Long gibberish (reward hacking!)
    ]
    
    rewards = [flawed_reward(t) for t in texts]
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    
    lengths = [len(t) for t in texts]
    colors = ['green', 'blue', 'red']
    labels = ['Good short', 'Good long', 'Gibberish (hacked!)']
    
    bars = ax.bar(labels, rewards, color=colors, alpha=0.7)
    ax.set_ylabel('Reward', fontsize=12)
    ax.set_title('Reward Hacking: Exploiting Length Bias', fontsize=14)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add length annotations
    for bar, length in zip(bars, lengths):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{length} chars',
                ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Reward Hacking Example ===")
    print("\nFlawed reward model just counts length!")
    print("\nPolicy learns to generate long gibberish for high reward.")
    print("\nMitigation strategies:")
    print("1. KL penalty: Keep policy close to reference model")
    print("2. Better reward model: Train on quality, not length")
    print("3. Ensemble: Use multiple reward models")
    print("4. Human-in-the-loop: Regular auditing")

simulate_reward_hacking()

## 7. Best Practices

### Data Collection
1. **Diverse prompts**: Cover full input distribution
2. **Multiple labelers**: Reduce individual bias
3. **Quality checks**: Filter low-agreement examples
4. **Balanced data**: Equal representation of domains

### Model Training
1. **Start from SFT model**: Better initialization
2. **Small learning rate**: Prevent catastrophic forgetting
3. **Early stopping**: Avoid overfitting
4. **Ensemble models**: Average multiple reward models

### Validation
1. **Hold-out test set**: Never train on validation data
2. **Human evaluation**: Regular quality checks
3. **Adversarial testing**: Look for failure modes
4. **Distribution analysis**: Check reward ranges

### Deployment
1. **Iterative updates**: Collect new preference data
2. **A/B testing**: Compare to baseline
3. **Monitoring**: Track reward distributions
4. **Fallbacks**: Detect and handle OOD inputs

In [ ]:
# Ensemble reward model
class EnsembleRewardModel(nn.Module):
    """Ensemble of multiple reward models for robustness."""
    
    def __init__(self, models: List[nn.Module], weights: Optional[List[float]] = None):
        super().__init__()
        self.models = nn.ModuleList(models)
        
        if weights is None:
            weights = [1.0 / len(models)] * len(models)
        self.weights = torch.tensor(weights)
    
    def forward(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Return weighted average of all models."""
        rewards = []
        
        for model in self.models:
            r = model(input_ids, attention_mask)
            rewards.append(r)
        
        rewards = torch.stack(rewards, dim=0)  # (num_models, batch_size)
        weights = self.weights.to(rewards.device).view(-1, 1)
        
        return (rewards * weights).sum(dim=0)
    
    def get_uncertainty(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Return standard deviation across models (epistemic uncertainty)."""
        rewards = []
        
        for model in self.models:
            r = model(input_ids, attention_mask)
            rewards.append(r)
        
        rewards = torch.stack(rewards, dim=0)
        return rewards.std(dim=0)


print("Ensemble Reward Model:")
print("- Combines multiple reward models")
print("- More robust to individual model failures")
print("- Provides uncertainty estimates")
print("- Can detect OOD inputs (high disagreement)")

## Summary

### Key Takeaways

1. **Reward modeling** is the first step in RLHF, learning human preferences from comparisons

2. **Bradley-Terry model** provides a principled probabilistic framework for ranking

3. **Architecture** is typically a language model + value head

4. **Training** uses preference comparisons with careful regularization

5. **Validation** requires multiple metrics: accuracy, distribution analysis, human agreement

6. **Failure modes** include reward hacking, overfitting, and distribution shift

7. **Best practices**: diverse data, ensemble models, iterative updates

### Next Steps

- **Notebook 41**: PPO algorithm for RLHF
- **Notebook 42**: DPO (simpler alternative to RLHF)
- **Notebook 43**: Constitutional AI

### References

1. Christiano et al. (2017): "Deep reinforcement learning from human preferences"
2. Stiennon et al. (2020): "Learning to summarize with human feedback"
3. Ouyang et al. (2022): "Training language models to follow instructions with human feedback"
4. Bai et al. (2022): "Training a Helpful and Harmless Assistant with RLHF"
5. Bradley & Terry (1952): "Rank Analysis of Incomplete Block Designs"